# Coca-Cola Company 10-K QA System
## Precision RAG with LlamaIndex — Last Decade of Annual Filings (2014–2023)

---

### System Architecture

| Stage | Component |
|---|---|
| **Ingestion** | LlamaParse — structured PDF → Markdown |
| **Retriever A** | Sentence Window Retriever |
| **Retriever B** | AutoMerging Retriever |
| **Retriever C** | Auto Retriever (metadata-aware) |
| **Post-Retrieval** | Cohere Reranker |
| **Query Engines** | Standard + SubQuestion Query Engine |
| **Evaluation** | Faithfulness + Relevancy (LLM-as-judge) |

### Data Source
Coca-Cola 10-K filings from SEC EDGAR (CIK: 0000021344) for fiscal years 2014–2023.

## 1. Environment Setup

In [ ]:
# Install all required packages
%pip install -qU llama-index-core
%pip install -qU llama-index-llms-openai
%pip install -qU llama-index-embeddings-openai
%pip install -qU llama-index-postprocessor-cohere-rerank
%pip install -qU llama-parse
%pip install -qU nest-asyncio requests beautifulsoup4 pandas matplotlib seaborn tqdm

In [ ]:
import os
import json
import time
import pickle
import asyncio
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nest_asyncio
from pathlib import Path
from typing import List, Dict, Optional
from tqdm import tqdm
from bs4 import BeautifulSoup

nest_asyncio.apply()

# LlamaIndex Core
from llama_index.core import (
    VectorStoreIndex,
    Settings,
    StorageContext,
    Document,
    load_index_from_storage,
)
from llama_index.core.node_parser import (
    SentenceWindowNodeParser,
    HierarchicalNodeParser,
    get_leaf_nodes,
)
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.indices.vector_store.retrievers import VectorIndexAutoRetriever
from llama_index.core.vector_stores.types import MetadataInfo, VectorStoreInfo
from llama_index.core.query_engine import RetrieverQueryEngine, SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

# LLM + Embeddings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# LlamaParse
from llama_parse import LlamaParse

# Cohere Reranker
from llama_index.postprocessor.cohere_rerank import CohereRerank

print("All imports successful.")

In [ ]:
# =============================================================
#  API KEYS — replace with your actual keys
# =============================================================
os.environ["OPENAI_API_KEY"]     = "sk-..."          # openai.com
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-..."         # cloud.llamaindex.ai
os.environ["COHERE_API_KEY"]     = "..."              # cohere.com

# Global model settings
Settings.llm         = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size  = 512
Settings.chunk_overlap = 50

print(f"LLM          : {Settings.llm.model}")
print(f"Embed model  : {Settings.embed_model.model_name}")
print(f"Chunk size   : {Settings.chunk_size}")

## 2. Download Coca-Cola 10-K Filings from SEC EDGAR

SEC EDGAR provides all 10-K filings publicly.  
Coca-Cola Company CIK: **0000021344**

In [ ]:
COCA_COLA_CIK = "0000021344"
SEC_HEADERS   = {"User-Agent": "RAG Research project@university.edu"}
DATA_DIR      = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def fetch_10k_filing_list(cik: str, start_year: int = 2014) -> List[Dict]:
    """Return metadata for all 10-K filings since start_year from SEC EDGAR."""
    url  = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    recent  = data["filings"]["recent"]
    forms   = recent["form"]
    dates   = recent["filingDate"]
    accnums = recent["accessionNumber"]

    filings = []
    for form, date, acc in zip(forms, dates, accnums):
        if form == "10-K" and int(date[:4]) >= start_year:
            filings.append(
                {
                    "fiscal_year": int(date[:4]) - 1,  # filed in Feb N+1 for year N
                    "filing_date": date,
                    "accession_number": acc,
                    "accession_clean": acc.replace("-", ""),
                }
            )

    # Deduplicate by fiscal year (keep earliest accession for each FY)
    seen = {}
    for f in sorted(filings, key=lambda x: x["filing_date"]):
        seen.setdefault(f["fiscal_year"], f)

    return sorted(seen.values(), key=lambda x: x["fiscal_year"])


def get_primary_document_url(cik_raw: str, accession: str, acc_clean: str) -> Optional[str]:
    """Find the primary 10-K document URL from the filing index."""
    cik_int = str(int(cik_raw))          # Strip leading zeros for URL paths
    idx_url  = (
        f"https://www.sec.gov/Archives/edgar/data/{cik_int}/"
        f"{acc_clean}/{accession}-index.json"
    )
    resp = requests.get(idx_url, headers=SEC_HEADERS, timeout=30)

    if resp.status_code == 200:
        idx_data = resp.json()
        for doc in idx_data.get("documents", []):
            doc_type = doc.get("type", "")
            doc_url  = doc.get("documentUrl", "")
            if "10-K" in doc_type and not doc_type.endswith("/A"):
                return "https://www.sec.gov" + doc_url

    # Fallback: parse the HTML index page
    html_idx_url = (
        f"https://www.sec.gov/Archives/edgar/data/{cik_int}/"
        f"{acc_clean}/{accession}-index.htm"
    )
    resp2 = requests.get(html_idx_url, headers=SEC_HEADERS, timeout=30)
    if resp2.status_code != 200:
        return None

    soup = BeautifulSoup(resp2.content, "html.parser")
    for row in soup.find_all("tr"):
        cols = row.find_all("td")
        if len(cols) >= 4:
            doc_type = cols[3].get_text(strip=True)
            link_tag  = cols[2].find("a")
            if link_tag and "10-K" in doc_type and not doc_type.endswith("/A"):
                href = link_tag.get("href", "")
                if href.startswith("/"):
                    return "https://www.sec.gov" + href
    return None


def download_filing(filing: Dict, save_dir: Path = DATA_DIR) -> Optional[Path]:
    """Download the primary 10-K document for a filing."""
    fy    = filing["fiscal_year"]
    acc   = filing["accession_number"]
    clean = filing["accession_clean"]

    doc_url = get_primary_document_url(COCA_COLA_CIK, acc, clean)
    if not doc_url:
        print(f"  [FY {fy}] Could not locate primary document.")
        return None

    ext      = ".pdf" if doc_url.lower().endswith(".pdf") else ".htm"
    out_file = save_dir / f"coca_cola_10k_{fy}{ext}"

    if out_file.exists():
        print(f"  [FY {fy}] Already downloaded: {out_file.name}")
        return out_file

    print(f"  [FY {fy}] Downloading from {doc_url} ...")
    doc_resp = requests.get(doc_url, headers=SEC_HEADERS, timeout=120)
    if doc_resp.status_code == 200:
        out_file.write_bytes(doc_resp.content)
        print(f"  [FY {fy}] Saved → {out_file.name} ({len(doc_resp.content)//1024} KB)")
        return out_file
    else:
        print(f"  [FY {fy}] HTTP {doc_resp.status_code} — skipping.")
        return None


print("Download utilities defined.")

In [ ]:
print("Fetching 10-K filing list from SEC EDGAR...")
filings = fetch_10k_filing_list(COCA_COLA_CIK, start_year=2015)  # filed from 2015→ covers FY 2014

print(f"Found {len(filings)} 10-K filings:\n")
for f in filings:
    print(f"  FY {f['fiscal_year']}  |  Filed: {f['filing_date']}  |  {f['accession_number']}")

print("\nDownloading documents...")
downloaded: List[Dict] = []

for filing in filings:
    path = download_filing(filing)
    if path:
        downloaded.append({"fiscal_year": filing["fiscal_year"], "path": path})
    time.sleep(0.4)  # Respect SEC rate limits

print(f"\nSuccessfully downloaded {len(downloaded)} filings.")

## 3. Parse Documents with LlamaParse

LlamaParse converts complex PDFs / HTML filings into clean Markdown,  
preserving tables and financial data layout.

> **Note:** Free tier = 1,000 pages/day. Results are cached locally after first parse.

In [ ]:
PARSE_CACHE = Path("parsed_documents.pkl")


def parse_documents_with_llamaparse(files: List[Dict]) -> List[Document]:
    """Parse each filing with LlamaParse and attach year metadata."""
    parser = LlamaParse(
        api_key=os.environ["LLAMA_CLOUD_API_KEY"],
        result_type="markdown",
        verbose=False,
        language="en",
        parsing_instruction=(
            "This is an annual 10-K report filed with the SEC. "
            "Extract all text including financial tables, footnotes, and risk factors. "
            "Preserve table structures using markdown formatting."
        ),
    )

    all_docs: List[Document] = []

    for item in tqdm(files, desc="Parsing with LlamaParse"):
        fy   = item["fiscal_year"]
        path = str(item["path"])

        try:
            docs = parser.load_data(path)
            for doc in docs:
                doc.metadata.update(
                    {
                        "year": fy,
                        "fiscal_year": str(fy),
                        "company": "The Coca-Cola Company",
                        "ticker": "KO",
                        "filing_type": "10-K",
                        "source": f"Coca-Cola 10-K FY{fy}",
                    }
                )
            all_docs.extend(docs)
            print(f"  FY {fy}: {len(docs)} pages parsed.")

        except Exception as exc:
            print(f"  FY {fy}: parse error — {exc}")

    return all_docs


if PARSE_CACHE.exists():
    print("Loading cached parsed documents...")
    with open(PARSE_CACHE, "rb") as fh:
        documents = pickle.load(fh)
    print(f"Loaded {len(documents)} document chunks from cache.")
else:
    documents = parse_documents_with_llamaparse(downloaded)
    with open(PARSE_CACHE, "wb") as fh:
        pickle.dump(documents, fh)
    print(f"Parsed and cached {len(documents)} document chunks.")

In [ ]:
# Document exploration
print(f"Total chunks  : {len(documents)}")
print(f"Years covered : {sorted(set(d.metadata['year'] for d in documents))}")
print(f"\nSample metadata:\n{documents[0].metadata}")
print(f"\nSample text (first 500 chars):\n{documents[0].text[:500]}")

# Distribution by year
year_counts = {}
for doc in documents:
    y = doc.metadata["year"]
    year_counts[y] = year_counts.get(y, 0) + 1

print("\nChunks per fiscal year:")
for y in sorted(year_counts):
    print(f"  FY {y}: {year_counts[y]} chunks")

---
## 4. Retriever A — Sentence Window Retriever

**Idea:** Index individual sentences (small, focused embeddings),  
but return a wider window of surrounding sentences at query time.  

| Step | Detail |
|---|---|
| Parsing | `SentenceWindowNodeParser(window_size=3)` — each node is one sentence |
| Storage | Window text stored in node metadata |
| Retrieval | Semantic match on sentences |
| Post-proc | `MetadataReplacementPostProcessor` swaps sentence → window |


In [ ]:
SW_PERSIST_DIR = Path("index_sentence_window")

sw_node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

if SW_PERSIST_DIR.exists():
    print("Loading persisted Sentence Window index...")
    sw_storage = StorageContext.from_defaults(persist_dir=str(SW_PERSIST_DIR))
    sw_index   = load_index_from_storage(sw_storage)
else:
    print("Building Sentence Window index (this may take a few minutes)...")
    sw_index = VectorStoreIndex.from_documents(
        documents,
        transformations=[sw_node_parser],
        show_progress=True,
    )
    sw_index.storage_context.persist(persist_dir=str(SW_PERSIST_DIR))
    print(f"Index persisted to {SW_PERSIST_DIR}")

# Retriever + post-processor
sw_retriever = sw_index.as_retriever(similarity_top_k=6)
sw_postproc  = MetadataReplacementPostProcessor(target_metadata_key="window")

# Query engine (no reranking yet)
sw_query_engine = RetrieverQueryEngine.from_args(
    retriever=sw_retriever,
    node_postprocessors=[sw_postproc],
)

print("Sentence Window query engine ready.")

In [ ]:
# Quick smoke test
sw_test_q = "What was Coca-Cola's net revenue in 2022?"
sw_resp    = sw_query_engine.query(sw_test_q)

print("[Sentence Window] Query:", sw_test_q)
print("Response:")
print(sw_resp)

---
## 5. Retriever B — AutoMerging Retriever

**Idea:** Build a hierarchy of chunks (coarse → fine).  
Index the finest level; merge back up toward the parent when many siblings are retrieved.

| Level | Chunk size |
|---|---|
| Root | 2048 tokens |
| Middle | 512 tokens |
| Leaf | 128 tokens ← **indexed** |

If **≥ 50%** of a parent's children are retrieved, the retriever replaces them with the parent.

In [ ]:
AM_PERSIST_DIR = Path("index_automerging")

am_node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128]
)

if AM_PERSIST_DIR.exists():
    print("Loading persisted AutoMerging index...")
    am_storage = StorageContext.from_defaults(persist_dir=str(AM_PERSIST_DIR))
    am_index   = load_index_from_storage(am_storage)
else:
    print("Building AutoMerging index...")
    am_nodes      = am_node_parser.get_nodes_from_documents(documents)
    am_leaf_nodes = get_leaf_nodes(am_nodes)
    print(f"  Total nodes : {len(am_nodes)}  |  Leaf nodes : {len(am_leaf_nodes)}")

    am_storage = StorageContext.from_defaults()
    am_storage.docstore.add_documents(am_nodes)   # store ALL levels

    am_index = VectorStoreIndex(
        am_leaf_nodes,                            # index only leaves
        storage_context=am_storage,
        show_progress=True,
    )
    am_index.storage_context.persist(persist_dir=str(AM_PERSIST_DIR))
    print(f"Index persisted to {AM_PERSIST_DIR}")

# Base retriever → AutoMerging retriever
am_base_retriever    = am_index.as_retriever(similarity_top_k=8)
automerging_retriever = AutoMergingRetriever(
    am_base_retriever,
    am_storage,
    verbose=True,
)

am_query_engine = RetrieverQueryEngine.from_args(
    retriever=automerging_retriever,
)

print("AutoMerging query engine ready.")

In [ ]:
am_test_q = "What were the main risk factors Coca-Cola identified in 2021?"
am_resp   = am_query_engine.query(am_test_q)

print("[AutoMerging] Query:", am_test_q)
print("Response:")
print(am_resp)

---
## 6. Retriever C — Auto Retriever (Metadata-Aware)

**Idea:** The LLM inspects the query, reads the `VectorStoreInfo` schema,  
and automatically generates the correct metadata **filter** before searching.

For example — *"What was Coca-Cola's operating income in 2020?"*  
→ LLM infers `year == 2020` filter → only FY 2020 chunks are searched.

This drastically reduces noise from wrong-year documents.

In [ ]:
AUTO_PERSIST_DIR = Path("index_auto")

if AUTO_PERSIST_DIR.exists():
    print("Loading persisted Auto Retriever index...")
    auto_storage = StorageContext.from_defaults(persist_dir=str(AUTO_PERSIST_DIR))
    auto_index   = load_index_from_storage(auto_storage)
else:
    print("Building Auto Retriever index...")
    auto_index = VectorStoreIndex.from_documents(
        documents,
        show_progress=True,
    )
    auto_index.storage_context.persist(persist_dir=str(AUTO_PERSIST_DIR))
    print(f"Index persisted to {AUTO_PERSIST_DIR}")

# Describe the vector store schema so the LLM can construct filters
vector_store_info = VectorStoreInfo(
    content_info=(
        "Annual 10-K reports of The Coca-Cola Company filed with the SEC. "
        "Covers financial performance, business strategy, risk factors, "
        "segment results, capital structure, and management commentary."
    ),
    metadata_info=[
        MetadataInfo(
            name="year",
            type="int",
            description=(
                "Fiscal year of the 10-K filing. "
                "Integer between 2014 and 2023 inclusive."
            ),
        ),
        MetadataInfo(
            name="fiscal_year",
            type="str",
            description="Fiscal year as a string, e.g. '2014', '2023'.",
        ),
        MetadataInfo(
            name="filing_type",
            type="str",
            description="Always '10-K' for annual reports.",
        ),
        MetadataInfo(
            name="ticker",
            type="str",
            description="Stock ticker, always 'KO'.",
        ),
    ],
)

auto_retriever = VectorIndexAutoRetriever(
    auto_index,
    vector_store_info=vector_store_info,
    similarity_top_k=6,
    verbose=True,
)

auto_query_engine = RetrieverQueryEngine.from_args(
    retriever=auto_retriever,
)

print("Auto Retriever query engine ready.")

In [ ]:
auto_test_q = "What was Coca-Cola's operating income in fiscal year 2020?"
auto_resp   = auto_query_engine.query(auto_test_q)

print("[Auto Retriever] Query:", auto_test_q)
print("Response:")
print(auto_resp)

---
## 7. Post-Retrieval Reranking — Cohere Rerank

After retrieval, a **cross-encoder** re-scores each candidate node against the query.  
This removes irrelevant chunks even if they had high cosine similarity.  

| Setting | Value |
|---|---|
| Model | `rerank-english-v3.0` |
| `top_n` | 3 (keep 3 best after reranking) |

In [ ]:
cohere_rerank = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"],
    top_n=3,
    model="rerank-english-v3.0",
)

# Sentence Window + Rerank
sw_rerank_engine = RetrieverQueryEngine.from_args(
    retriever=sw_retriever,
    node_postprocessors=[sw_postproc, cohere_rerank],
)

# AutoMerging + Rerank
am_rerank_engine = RetrieverQueryEngine.from_args(
    retriever=automerging_retriever,
    node_postprocessors=[cohere_rerank],
)

# Auto Retriever + Rerank
auto_rerank_engine = RetrieverQueryEngine.from_args(
    retriever=auto_retriever,
    node_postprocessors=[cohere_rerank],
)

print("Reranked query engines ready.")

# Demonstrate reranking effect on one query
demo_q = "How did the COVID-19 pandemic impact Coca-Cola's 2020 revenue?"

print("\n--- Without rerank ---")
print(sw_query_engine.query(demo_q))

print("\n--- With Cohere rerank ---")
print(sw_rerank_engine.query(demo_q))

---
## 8. Query Engine Configuration

A `RetrieverQueryEngine` wires a retriever → (optional post-processors) → synthesizer.  
The synthesizer sends the retrieved context + question to the LLM to produce the final answer.

We build **six query engines** (3 retrievers × 2 reranking modes) for comparison.

In [ ]:
ENGINES = {
    "SentenceWindow"            : sw_query_engine,
    "SentenceWindow+Rerank"     : sw_rerank_engine,
    "AutoMerging"               : am_query_engine,
    "AutoMerging+Rerank"        : am_rerank_engine,
    "AutoRetriever"             : auto_query_engine,
    "AutoRetriever+Rerank"      : auto_rerank_engine,
}

print("Configured query engines:")
for name in ENGINES:
    print(f"  {name}")

# Side-by-side comparison on a sample question
compare_q = "What was Coca-Cola's total revenue and net income in 2023?"
print(f"\nQuestion: {compare_q}\n" + "="*60)

for name, engine in ENGINES.items():
    try:
        resp = engine.query(compare_q)
        print(f"\n[{name}]")
        print(str(resp)[:400])
    except Exception as e:
        print(f"\n[{name}] ERROR: {e}")

---
## 9. SubQuestion Query Engine

**Idea:** Decompose complex multi-part questions into simpler sub-questions,  
route each sub-question to the most relevant per-year engine,  
then synthesize a unified answer.

```
User question
     │
     ▼
  Sub-question generator
  ┌────┬────┬────┐
  │FY14│FY18│FY23│  ← per-year query engines
  └────┴────┴────┘
     │
     ▼
  Synthesizer → final answer
```

In [ ]:
SUBQ_PERSIST_DIR = Path("index_subq_years")
SUBQ_PERSIST_DIR.mkdir(exist_ok=True)

print("Building per-year query engines for SubQuestion engine...")

year_tools = []
all_years  = sorted(set(d.metadata["year"] for d in documents))

for year in all_years:
    year_docs = [d for d in documents if d.metadata["year"] == year]
    if not year_docs:
        continue

    persist_path = SUBQ_PERSIST_DIR / str(year)

    if persist_path.exists():
        yr_storage = StorageContext.from_defaults(persist_dir=str(persist_path))
        yr_index   = load_index_from_storage(yr_storage)
    else:
        yr_index = VectorStoreIndex.from_documents(year_docs, show_progress=False)
        yr_index.storage_context.persist(persist_dir=str(persist_path))

    yr_engine = yr_index.as_query_engine(
        similarity_top_k=4,
        node_postprocessors=[cohere_rerank],
    )

    year_tools.append(
        QueryEngineTool(
            query_engine=yr_engine,
            metadata=ToolMetadata(
                name=f"coca_cola_10k_{year}",
                description=(
                    f"Annual 10-K report of The Coca-Cola Company for fiscal year {year}. "
                    f"Use this tool for questions specifically about the {year} fiscal year: "
                    f"financial results, strategy, risks, operations, and outlook."
                ),
            ),
        )
    )
    print(f"  FY {year}: {len(year_docs)} chunks → engine ready.")

# Also add an all-years engine as a fallback tool
year_tools.append(
    QueryEngineTool(
        query_engine=sw_rerank_engine,
        metadata=ToolMetadata(
            name="coca_cola_10k_all_years",
            description=(
                "Searches across ALL Coca-Cola 10-K filings from 2014 to 2023. "
                "Use for cross-year trend questions or when no specific year is mentioned."
            ),
        ),
    )
)

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=year_tools,
    use_async=True,
    verbose=True,
)

print(f"\nSubQuestion engine ready with {len(year_tools)} tools.")

In [ ]:
# Test 1 — multi-year trend question
subq_q1 = (
    "How did Coca-Cola's net revenue change from 2019 to 2023? "
    "What were the key drivers of this change?"
)
print("SubQuestion Query 1:", subq_q1)
subq_resp1 = sub_question_engine.query(subq_q1)
print("\nFinal Answer:")
print(subq_resp1)

In [ ]:
# Test 2 — comparative analysis
subq_q2 = (
    "Compare Coca-Cola's operating margin in 2018 versus 2022. "
    "What acquisitions or divestitures affected their portfolio during this period?"
)
print("SubQuestion Query 2:", subq_q2)
subq_resp2 = sub_question_engine.query(subq_q2)
print("\nFinal Answer:")
print(subq_resp2)

---
## 10. Evaluation — Faithfulness & Relevancy

**LLM-as-judge evaluation** using two metrics:

| Metric | Measures |
|---|---|
| **Faithfulness** | Are all claims in the answer supported by the retrieved context? |
| **Relevancy** | Does the answer actually address the question asked? |

Both return a score in **[0, 1]**.

In [ ]:
EVAL_QUESTIONS = [
    # Financial performance
    "What was Coca-Cola's total net revenue for fiscal year 2023?",
    "What was Coca-Cola's net income in 2022?",
    "How did Coca-Cola's gross profit margin change from 2019 to 2023?",
    "What was Coca-Cola's capital expenditure in 2021?",
    "What dividends per share did Coca-Cola pay in 2020?",

    # Operations and strategy
    "How did the COVID-19 pandemic affect Coca-Cola's business in 2020?",
    "What were Coca-Cola's main product categories as described in the 2022 10-K?",
    "Which geographic segments contributed most to revenue in 2021?",
    "What acquisitions did Coca-Cola complete in 2018?",
    "How did Coca-Cola restructure its bottling operations between 2015 and 2020?",

    # Risk factors
    "What cybersecurity risks did Coca-Cola disclose in its 2023 10-K filing?",
    "What foreign currency risks were highlighted in Coca-Cola's 2022 annual report?",
    "How did raw material cost pressures affect Coca-Cola in 2022?",

    # Cross-year trends
    "How did Coca-Cola's cash and equivalents change from 2014 to 2023?",
    "What was Coca-Cola's long-term debt level in 2023 compared to 2018?",
]

print(f"Evaluation question set: {len(EVAL_QUESTIONS)} questions.")
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"  {i:02d}. {q}")

In [ ]:
eval_llm = OpenAI(model="gpt-4o-mini", temperature=0)

faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_llm)
relevancy_evaluator    = RelevancyEvaluator(llm=eval_llm)


def evaluate_engine(engine, questions: List[str], engine_name: str) -> List[Dict]:
    """Run faithfulness + relevancy evaluation on a query engine."""
    results = []
    for q in tqdm(questions, desc=f"Evaluating {engine_name}"):
        try:
            response = engine.query(q)

            faith_result = faithfulness_evaluator.evaluate_response(response=response)
            rel_result   = relevancy_evaluator.evaluate_response(
                query=q, response=response
            )

            results.append(
                {
                    "engine"      : engine_name,
                    "question"    : q,
                    "answer"      : str(response),
                    "faithfulness": faith_result.score if faith_result.score is not None else 0.0,
                    "relevancy"   : rel_result.score   if rel_result.score   is not None else 0.0,
                }
            )
        except Exception as exc:
            print(f"  Error on '{q[:50]}...' with {engine_name}: {exc}")
            results.append(
                {
                    "engine"      : engine_name,
                    "question"    : q,
                    "answer"      : "[ERROR]",
                    "faithfulness": 0.0,
                    "relevancy"   : 0.0,
                }
            )
    return results


print("Evaluators configured.")

In [ ]:
EVAL_RESULTS_CACHE = Path("eval_results.pkl")

if EVAL_RESULTS_CACHE.exists():
    print("Loading cached evaluation results...")
    with open(EVAL_RESULTS_CACHE, "rb") as fh:
        all_eval_results = pickle.load(fh)
else:
    all_eval_results = []

    # Engines to evaluate (names must match ENGINES dict)
    eval_engines = {
        "SentenceWindow"        : sw_query_engine,
        "SentenceWindow+Rerank" : sw_rerank_engine,
        "AutoMerging"           : am_query_engine,
        "AutoMerging+Rerank"    : am_rerank_engine,
        "AutoRetriever"         : auto_query_engine,
        "AutoRetriever+Rerank"  : auto_rerank_engine,
        "SubQuestion"           : sub_question_engine,
    }

    for eng_name, engine in eval_engines.items():
        results = evaluate_engine(engine, EVAL_QUESTIONS, eng_name)
        all_eval_results.extend(results)
        print(f"  {eng_name}: done ({len(results)} questions)")

    with open(EVAL_RESULTS_CACHE, "wb") as fh:
        pickle.dump(all_eval_results, fh)
    print("Results cached.")

eval_df = pd.DataFrame(all_eval_results)
print(f"\nTotal evaluations: {len(eval_df)}")
eval_df.head()

---
## 11. Results & Visualization

In [ ]:
avg_scores = (
    eval_df.groupby("engine")[["faithfulness", "relevancy"]]
    .mean()
    .sort_values("relevancy", ascending=False)
)
avg_scores["combined"] = (avg_scores["faithfulness"] + avg_scores["relevancy"]) / 2

print("Average Scores by Engine:")
print(avg_scores.round(3).to_string())

# Grouped bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = sns.color_palette("husl", len(avg_scores))

metrics = ["faithfulness", "relevancy", "combined"]
titles  = ["Faithfulness Score", "Relevancy Score", "Combined Score"]

for ax, metric, title in zip(axes, metrics, titles):
    bars = avg_scores[metric].plot(
        kind="barh", ax=ax, color=palette, edgecolor="white"
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Score (0–1)", fontsize=11)
    ax.set_xlim(0, 1.05)
    ax.axvline(x=0.7, color="red", linestyle="--", linewidth=1, alpha=0.6, label="0.7 threshold")
    for i, v in enumerate(avg_scores[metric]):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle(
    "Coca-Cola 10-K RAG — Retriever Performance Comparison",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig("retriever_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → retriever_comparison.png")

In [ ]:
# Heatmap: per-question faithfulness by engine
pivot_faith = eval_df.pivot_table(
    index="question", columns="engine", values="faithfulness"
)
# Shorten question labels for display
pivot_faith.index = [q[:55] + "..." if len(q) > 55 else q for q in pivot_faith.index]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot_faith,
    annot=True, fmt=".2f",
    cmap="RdYlGn",
    vmin=0, vmax=1,
    linewidths=0.4,
    ax=ax,
)
ax.set_title(
    "Faithfulness Heatmap — Question × Engine",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Engine", fontsize=11)
ax.set_ylabel("Question", fontsize=11)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("faithfulness_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Reranking impact: compare each retriever with/without rerank
rerank_pairs = [
    ("SentenceWindow",   "SentenceWindow+Rerank"),
    ("AutoMerging",      "AutoMerging+Rerank"),
    ("AutoRetriever",    "AutoRetriever+Rerank"),
]

rerank_delta_rows = []
for base_name, rerank_name in rerank_pairs:
    if base_name in avg_scores.index and rerank_name in avg_scores.index:
        delta_f = avg_scores.loc[rerank_name, "faithfulness"] - avg_scores.loc[base_name, "faithfulness"]
        delta_r = avg_scores.loc[rerank_name, "relevancy"]    - avg_scores.loc[base_name, "relevancy"]
        rerank_delta_rows.append(
            {
                "Retriever"             : base_name,
                "Δ Faithfulness (rerank)": delta_f,
                "Δ Relevancy (rerank)"  : delta_r,
            }
        )

delta_df = pd.DataFrame(rerank_delta_rows).set_index("Retriever")
print("Reranking Impact (positive = improvement):")
print(delta_df.round(3).to_string())

delta_df.plot(
    kind="bar", figsize=(9, 5),
    color=["steelblue", "coral"],
    edgecolor="white",
)
plt.title("Reranking Delta per Retriever", fontsize=13, fontweight="bold")
plt.ylabel("Score Delta")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("reranking_delta.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Faithfulness vs Relevancy scatter per engine
fig, ax = plt.subplots(figsize=(9, 6))
colors  = sns.color_palette("tab10", n_colors=eval_df["engine"].nunique())

for (eng, grp), color in zip(eval_df.groupby("engine"), colors):
    ax.scatter(
        grp["faithfulness"], grp["relevancy"],
        label=eng, alpha=0.7, s=60, color=color,
    )
    # Mark centroid
    ax.scatter(
        grp["faithfulness"].mean(), grp["relevancy"].mean(),
        marker="*", s=220, color=color, edgecolors="black", linewidths=0.8,
    )

ax.set_xlabel("Faithfulness", fontsize=12)
ax.set_ylabel("Relevancy",    fontsize=12)
ax.set_title(
    "Faithfulness vs Relevancy (★ = centroid)",
    fontsize=13, fontweight="bold",
)
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.axhline(0.7, color="gray", linestyle="--", alpha=0.5)
ax.axvline(0.7, color="gray", linestyle="--", alpha=0.5)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("faithfulness_vs_relevancy.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 12. Final Summary

In [ ]:
print("="*65)
print(" COCA-COLA 10-K RAG SYSTEM — FINAL RESULTS SUMMARY")
print("="*65)
print(f"\nDocuments parsed    : {len(documents)} chunks")
print(f"Fiscal years covered: {sorted(set(d.metadata['year'] for d in documents))}")
print(f"Evaluation questions: {len(EVAL_QUESTIONS)}")
print(f"Engines evaluated   : {eval_df['engine'].nunique()}")

print("\n" + "-"*65)
print(" AVERAGE SCORES  (faithfulness / relevancy / combined)")
print("-"*65)

summary = avg_scores[["faithfulness", "relevancy", "combined"]].copy()
summary.columns = ["Faithfulness", "Relevancy", "Combined"]
print(summary.round(4).to_string())

best_engine = avg_scores["combined"].idxmax()
best_score  = avg_scores.loc[best_engine, "combined"]

print("\n" + "-"*65)
print(f" BEST ENGINE: {best_engine}  (combined score: {best_score:.4f})")
print("-"*65)

print("""
KEY FINDINGS
─────────────────────────────────────────────────────────────────
• Sentence Window captures precise financial figures via focused
  sentence embeddings; context window prevents answer truncation.

• AutoMerging excels at broad narrative questions (risk factors,
  strategy) by merging sibling chunks into coherent paragraphs.

• Auto Retriever provides the best year-specific precision thanks
  to LLM-generated metadata filters that eliminate cross-year noise.

• Cohere Reranking consistently improves both metrics by re-scoring
  retrieved chunks with a cross-encoder rather than cosine similarity.

• SubQuestion Engine handles multi-year trend analysis best by
  decomposing complex questions and routing to per-year engines.
─────────────────────────────────────────────────────────────────
""")

In [ ]:
# Interactive QA demo using the best engine
best_engine_obj = {
    "SentenceWindow"        : sw_query_engine,
    "SentenceWindow+Rerank" : sw_rerank_engine,
    "AutoMerging"           : am_query_engine,
    "AutoMerging+Rerank"    : am_rerank_engine,
    "AutoRetriever"         : auto_query_engine,
    "AutoRetriever+Rerank"  : auto_rerank_engine,
    "SubQuestion"           : sub_question_engine,
}[best_engine]

DEMO_QUESTIONS = [
    "What was Coca-Cola's total net revenue in 2023 and how did it compare to 2019?",
    "Describe Coca-Cola's approach to sustainability and ESG in their recent 10-K filings.",
    "What was the impact of inflation on Coca-Cola's cost of goods sold in 2022?",
]

print(f"DEMO — Using best engine: {best_engine}")
print("="*60)

for dq in DEMO_QUESTIONS:
    print(f"\nQ: {dq}")
    print("-"*60)
    ans = best_engine_obj.query(dq)
    print(f"A: {ans}")

---
## Appendix — Architecture Diagram

```
Coca-Cola 10-K PDFs (FY 2014–2023)
            │
            ▼
       LlamaParse
    (Markdown output)
            │
  ┌─────────┼──────────┐
  │         │          │
  ▼         ▼          ▼
Sentence  AutoMerge  Standard
 Window    Parser     Index
 Parser   (3 levels)
  │         │          │
  ▼         ▼          ▼
 SW       AM        Auto
Index    Index    Retriever
(vec)   (leaf)   (metadata)
  │         │          │
  └────┬────┘──────────┘
       │
       ▼
  Cohere Rerank
  (cross-encoder)
       │
  ┌────┴─────────────────┐
  │                      │
  ▼                      ▼
RetrieverQueryEngine  SubQuestion
 (per retriever)       QueryEngine
       │               (per year)
       └────────┬───────────┘
                ▼
    GPT-4o-mini Synthesizer
                │
                ▼
        Final Answer

Evaluation:
  FaithfulnessEvaluator + RelevancyEvaluator
  (LLM-as-judge, gpt-4o-mini)
```

---
*Submission: zip this `.ipynb` and upload as final submission.*